##Ⅰ. 데이터 정제 및 공통작업

###1. 두개의 data를 합치고 Null 이 존재하는 행 제거
####가. 필요한 솔루션 설치

In [ ]:
!pip uninstall lark
!pip install lark

In [ ]:
# 다운로드 데이터를 사용하기 위해 구글드라이브와 연결한다.
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab\ Notebooks/testdata/contest/
%ls

In [ ]:
!pip install -qU langchain langchain-openai langchain_community

In [ ]:
# API 키는 코드에 직접 적지 않는다.
# Colab 좌측 '보안 비밀(🔑)'에 OPENAI_API_KEY 를 등록하거나, 환경변수로 설정한 뒤 실행한다.
import os

try:
    from google.colab import userdata
    OPENAI_KEY = userdata.get("OPENAI_API_KEY")
except Exception:
    OPENAI_KEY = os.environ["OPENAI_API_KEY"]

os.environ["OPENAI_API_KEY"] = OPENAI_KEY

In [ ]:
# 벡터DB로 사용할 chroma DBMS 설치
!pip install  -U  langchain-chroma

In [ ]:
!pip install openpyxl

In [ ]:
!pip install sentence-transformers

####나. 임베딩 모델 설정
* 둘중 어느 모델을 사용할지는 vectorstore 에서 결정

In [ ]:
from langchain.embeddings import OpenAIEmbeddings

model_ada = OpenAIEmbeddings(
    model="text-embedding-3-large",
    openai_api_key=OPENAI_KEY
)

embedding_name = model_ada

In [ ]:
# 임베딩 모듈 다운로드 (한국어 KorNLU 에 학습시킨 모델임)

from langchain.embeddings import HuggingFaceEmbeddings

model_name = "jhgan/ko-sroberta-multitask" # (KorNLU 데이터셋에 학습시킨 한국어 임베딩 모델)
model_kwargs = {'device': 'cpu'}          # cpu 또는 gpu 환경에서는 cuda 를 지정한다.
encode_kwargs = {'normalize_embeddings': False}
model_huggingface = HuggingFaceEmbeddings(
    model_name=model_name,
    model_kwargs=model_kwargs,
    encode_kwargs=encode_kwargs
)

embedding_name = model_huggingface

####다. 메인 데이터 생성작업 (만약 vectordb를 저장해 놓았을 경우에는 이작업은 SKIP 한다)

In [ ]:
import pandas as pd

# 학습용 데이터 를 판다스의 DataFrame 형태로 로딩하여 train_data 에 저장 및 개수 출력
train_data = pd.read_excel('DS학술제-모델링경진대회_Train.xlsx')
print("학습용 데이터 갯수 : ", len(train_data))

In [ ]:
# 검증용 데이터를 판다스의 DataFrame 형태로 로딩하여 valid_data 에 저장 및 개수 출력
valid_data = pd.read_excel('DS학술제-모델링경진대회_Valid.xlsx')
print("학습용 데이터 갯수 : ", len(valid_data))

In [ ]:
# 두개의 데이터를 합친다.
tot_data = pd.concat([train_data, valid_data])

In [ ]:
# 데이터의 총 행의 숫자를 출력한다.
print('NULL 삭제전 데이터의 개수 :',len(tot_data))

In [ ]:
# tot_data 의 values 필드에 null 이 존재하는지 점검
# 컬럼중에 null 이 하나라도 존재하는 개수를 출력함
print('NULL 값 존재 유무 :', tot_data.isnull().values.any())

In [ ]:
# 만약 Null 값이 존재하면 아래의 프로그램을 실행하여 Null 을 제거한다.
# Null 값이 존재하는 행 제거, any 는 무조건 지우라는 의미를 가진다.
# train_data = train_data.dropna(how = 'any')
# print('NULL 값 존재 유무 :', train_data.isnull().values.any()) # Null 값이 존재하는지 확인
# print('NULL 삭제 후 데이터의 개수 :',len(train_data))

In [ ]:
tot_data[:2]

* Dup 항목을 제거하여 하나로 통합

In [ ]:
print('중복데이터 삭제전 데이터의 개수 :',len(tot_data))

In [ ]:
# 중복데이터를 제거한다.
tot_data = tot_data.drop_duplicates()

In [ ]:
print('중복데이터 삭제 후 데이터의 개수 :',len(tot_data))

##Ⅲ. 벡터 환경 구성

###1. 벡터DB 및 retriever 생성

#### 주의사항
* 1.  만약 요약정보를 멀티벡터를 구성할 경우
  > -  dbsave_dir = "./store/summarise/"             <BR>
  > -  collection_name = "summaries"
* 2.  만약 Chunk 로 멀티벡터를 구성할 경우  
  > -  dbsave_dir = "./store/smallizer/"             <BR>
  > -  collection_name = "smallizer"
* 3. 메타로 구성할 경우
  > -  dbsave_dir = "./store/meta/"
  > -  collection_name = "meta"   

In [ ]:
!pip show langchain-chroma

* 만약 디스크에 저정되어 있지 않다면 빈 벡터DB를 생성하고
* 디스크에 저장되어 있다면 이를 읽어 온다.

In [ ]:
# [선택] 벡터DB를 처음부터 다시 만들고 싶을 때만 사용한다.
# 아래 셀에서 vectorstore01 ~ 03 을 생성한 뒤, 초기화할 컬렉션의 주석을 해제하고 실행한다.
# vectorstore01.delete_collection()   # 방법1: summarise
# vectorstore02.delete_collection()   # 방법2: smallize
# vectorstore03.delete_collection()   # 방법3: meta

In [ ]:
from langchain.vectorstores import Chroma
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import LocalFileStore

dbsave_dir = "./store/summarise/"
collection_name = "summarise"

# ChromaDB 를 벡터 스토어로 생성하였다.
# 위에서 생성한 임베딩 모델 중 한가지를 선택하여 아래의 작업을 실행한다.
vectorstore01 = Chroma(
    collection_name=collection_name,
    persist_directory=dbsave_dir,
    embedding_function=model_huggingface
)

In [ ]:
from langchain.vectorstores import Chroma
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import LocalFileStore

dbsave_dir = "./store/samllize/"
collection_name = "smallize"

# ChromaDB 를 벡터 스토어로 생성하였다.
# 위에서 생성한 임베딩 모델 중 한가지를 선택하여 아래의 작업을 실행한다.
vectorstore02 = Chroma(
    collection_name=collection_name,
    persist_directory=dbsave_dir,
    embedding_function=model_huggingface
)

In [ ]:
from langchain.vectorstores import Chroma
from langchain.retrievers.multi_vector import MultiVectorRetriever
from langchain.storage import LocalFileStore

dbsave_dir = "./store/meta/"
collection_name = "meta"

# ChromaDB 를 벡터 스토어로 생성하였다.
# 위에서 생성한 임베딩 모델 중 한가지를 선택하여 아래의 작업을 실행한다.
vectorstore03 = Chroma(
    collection_name=collection_name,
    persist_directory=dbsave_dir,
    embedding_function=model_huggingface
)

#### 2. 데이터 토큰나이저 및 UUID 키 지정 (만약 벡터 DB 파일로 저장되어 있다면 이 작업은 SKIP 함)

In [ ]:
from langchain.storage import InMemoryByteStore
from langchain_chroma import Chroma
from langchain_community.document_loaders import TextLoader
from langchain_openai import OpenAIEmbeddings
from langchain_text_splitters import RecursiveCharacterTextSplitter

In [ ]:
출원번호 = [str(column) for column in tot_data["출원번호"]]

In [ ]:
메인IPC2 = [str(column) for column in tot_data["메인IPC2"]]

In [ ]:
전체IPC = [str(column) for column in tot_data["전체 IPC"]]

In [ ]:
등록번호 = [str(column) for column in tot_data["등록번호"]]

In [ ]:
출원인 = [str(column) for column in tot_data["출원인"]]

In [ ]:
발명자 = [str(column) for column in tot_data["발명자"]]

In [ ]:
법적상태 = [str(column) for column in tot_data["법적상태"]]

In [ ]:
import uuid
doc_ids = [str(uuid.uuid4()) for _ in range(len(tot_data))]
doc_ids

###3. 전체 문서를 가공함  (만약 벡터 DB 파일로 저장되어 있다면 이 작업은 SKIP 함)

In [ ]:
from langchain.docstore.document import Document

docs = []
id_key = "doc_id"
count = 0

# DataFrame 으로 부터 데이터를 한쭐씩 로딩한다.
for index, doc in tot_data.iterrows():
    line = ""
    line = ' ;'.join([column + " : " + str(doc[column]) for column in tot_data.columns])
    docs.append(Document(page_content = line, ids=doc_ids[count], metadata={id_key: doc_ids[count], "메인IPC2" : 메인IPC2[count], "전체 IPC" : 전체IPC[count], "출원번호" : 출원번호[count], \
                                                                                                    "등록번호" : 등록번호[count], "출원인" : 출원인[count],"발명자" : 발명자[count], \
                                                                                                    "법적상태" : 법적상태[count] }))
    count = count + 1

In [ ]:
docs[0]

In [ ]:
len(docs)

##Ⅳ. 데이터 저장 → 벡터DB 생성 → Retriever 생성 → 체인 생성

### 아래의 방법1 ~ 방법3 중 하나를 선택하여 정보를 추출한다.

### 1. (방법1)요약 정보로 멀티벡터 DB 생성 <br>
      (만약 벡터 DB가 존재한다면 저장되어 로딩 되었다면 가. ~ 마. 작업은 SKIP)

####가. 구글드라이브 설정 (이미 벡터DB 데이터가 존재할 경우 가. ~ 라. SKIP)

In [ ]:
# 다운로드 데이터를 사용하기 위해 구글드라이브와 연결한다.
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab\ Notebooks/testdata/contest/
%ls

####나. 요약정보 생성작업 (요약 자료가 없는 경우 실행)

* 만약 요약 문서 정보가 존재하지 않을 경우 아래를 실행한다.
* 하지만, 이미 아래 요약 명령을 실행하여 요약 데이터가 존재할 경우에는 <br>
  나. 를 생략하고 요약 정보를 가져오는 다. 를 실행한다.

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_openai import ChatOpenAI
from langchain_core.output_parsers import StrOutputParser

chain = (
    # doc 은 딕셔너리나 스트링이 아니다.
    # doc 이 [Document] 즉 Document 의 리스트 임에도 동작하는 것에 주의할 것
    {"doc": lambda x: x.page_content}
    # 문서 요약을 위한 프롬프트 템플릿 생성
    | ChatPromptTemplate.from_messages(
        [
            ("system", "You are an expert in summarizing documents in Korean."),
            (
                "user",
                "Summarize the following documents in 3 sentences in bullet points format.\n\n{doc}",
            ),
        ]
    )
    # OpenAI의 ChatGPT 모델을 사용하여 요약 생성
    | ChatOpenAI(temperature=0, model="gpt-4o-mini", openai_api_key=OPENAI_KEY)
    | StrOutputParser()
)

In [ ]:
# batch 메서드를 사용하여 docs 리스트의 문서들을 일괄 요약하도록 한다.
# max_concurrecy 를 5로 설정하여 최대 5개의 문서를 동시에 처리하도록 한다.

id_key = "doc_id"

summaries = chain.batch(docs, {"max_concurrency": 5})
summaries

In [ ]:
print(len(docs))
print(len(summaries))

In [ ]:
import pickle

with open("list.pickle","wb") as f:
  pickle.dump(summaries, f)

####다. 이미 요약자료를 생성해서 이미 저장한 경우 이를 읽어 온다.
* 디스크에 요약 정보가 저장되어 있을 경우에만 실행한다.

In [ ]:
# 다운로드 데이터를 사용하기 위해 구글드라이브와 연결한다.
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab\ Notebooks/testdata/contest/
%ls

In [ ]:
import pickle

with open("list.pickle","rb") as f:
   summaries = pickle.load(f)

In [ ]:
len(summaries)

####라. 요약정보를 바탕으로 SUB 벡터를 생성한다.

In [ ]:
docs[:2]

In [ ]:
id_key

In [ ]:
# docs 의 갯수와 summeries 의 갯수가 같기 때문에 doc_ids[i] 를 직접 대입할 수 있다.
id_key = "doc_id"
sub_docs = [Document(page_content=(s + "; 메인IPC2 : " + 메인IPC2[i] +  "; 전체 IPC : " + 전체IPC[i] + "; 출원번호 : " + 출원번호[i] \
                                     + "; 등록번호 : " + 등록번호[i] + "; 출원인 : " + 출원인[i]  + "; 발명자 : " + 발명자[i] + "; 법적상태 : " + 법적상태[i]), id_key = doc_ids[i],\
                     metadata={id_key: doc_ids[i], "메인IPC2": 메인IPC2[i], "전체 IPC": 전체IPC[i], "출원번호": 출원번호[i]}) for i, s in enumerate(summaries)]

In [ ]:
sub_docs[:2]

####마. 멀티 Vector Retriever 를 생성하고 데이터를 저장한다.

In [ ]:
# vector db 를 메모리에 저장한다.
# Large Chunk 를 저장할 것이다.
# =====> 이부분 store 에 디렉토리를 지정하면 디스크에 저장됨

# 혹시 존재할 수 있는 기존 데이터를 모두 삭제한다.
#
# 만약 메모리에 vector DB를 사용하려면 InMemoryByteStore()을 사용하고
# 디스크를 사용하려면 LocalFileStore() 를 사용한다.
# store = InMemoryByteStore()
store = LocalFileStore("./store/summarise")

id_key = "doc_id"

# [중요] 일반 Retriever 가 아니라 MultiVectorRetriever 를 선언하였다.
# - vectorstore 또는 docstore 에는 Large Chunk 를 저장하고
# - byte_store 에는 Small Chuhk 를 저장한다.
retriever01 = MultiVectorRetriever(
    vectorstore=vectorstore01,         # 자식 문서 즉 작은 Chunk가 저장된다.
    persist_directory="./store/summarise",    # 만약 디스크에 영구 보관하고 싶다면 아래와 같이 지정한다.
    byte_store=store,                # 부모 문서 즉 큰 Chunk가 저장된다.
    id_key=id_key,                   # id_key 를 지정하여 상호 연결되도록 구성함
    k=3
)

# 만약 BYTE 단위가 아니라 DOC 단위로 저장하고 싶다면
# store = InMemoryStore()
# retriever = ParentDocumentRetriever(
#    vectorstore=vectorstore,
#    docstore=store,
#    child_splitter=child_splitter,
#    parent_splitter=parent_splitter
# )

# 일반적으로 사용했던
# vectorstore = Chroma.from_documents(documents=docs, embedding=embedding_model)
# retriever = vectorstore.as_retriever()
# 도 같이 기억해 둘것

####바. 데이터를 저장한다. (만약 벡터DB에 이미 데이터가 존재할 경우 SKIP)

In [ ]:
# 작게 쪼갠 즉 400단위로 쪼갠 벡터를 vectstore 에 저장한다.
# 벡터 스토어는 아래와 같이 위에 정의되어 있다.
# vectorstore = Chroma(
#    collection_name="full_documents", embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_KEY)
# )

retriever01.vectorstore.add_documents(sub_docs)

# mset() 메서드를 통해 문서ID 와 문서 내용을 KEY-VALUE 쌍으로 문서 저장소에 저장한다.
# docstore 로 저장하게 되면 byte_store 로 지정된 영역에 저장된다.
retriever01.docstore.mset(list(zip(doc_ids, docs)))

####바. 요약정보를 가지고 MultiVector 로 구성한 retriever 의 정확성 검증

In [ ]:
retriever01.invoke("'지능정보 기술을 활용하여 사업장 내 유해인자 발생에 대한 정보를 제공하고, 서비스와 근로자의 질병위험도를 예측할 수 있는' 와 관련 있는 특허를 알려주세요")

In [ ]:
retriever01.invoke("전체 IPC 가 'G16H-010/60,\[G06Q-010/10, G16H-010/20, G16H-080/00\]' 특허를 알려주세요")

####사. Retriever Agent 생성

In [ ]:
from langchain_openai import ChatOpenAI

# ChatOpenAI 클래스를 langchain_openai 모듈에서 가져옵니다.
llm = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0, openai_api_key=OPENAI_KEY)

In [ ]:
# 사용자 정의가 아닌 외부 도구에 의한 TOOL 은 아래와 같이 정의한다.

from langchain.tools.retriever import create_retriever_tool

retriever_tool01 = create_retriever_tool(
    retriever01,
    # 함수이름은 반드시 영문자로 해야 한다. 한글로 하면 에러가 발생한다.
    name="Patent_inquiry01",
    description="사용자의 특허 관련 질문을 검색합니다. 만약 여기에 존재하지 않는 특허와 관련한 질문이라면 LLM을 통해 \
검색합니다. 만약 사용자의 질문에 IPC가 포함되어 있다면 반드시 여기에서 얻은 답변에도 동일한 IPC가 포함되어야 합니다."
)

In [ ]:
# tools 리스트에 search와 retriever_tool을 추가합니다.
tools01 = [retriever_tool01]

In [ ]:
from langchain import hub
from langchain.prompts.chat import SystemMessagePromptTemplate

# hub에서 prompt를 가져옵니다 - 이 부분을 수정할 수 있습니다!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.append(SystemMessagePromptTemplate.from_template("만약 사용자의 질문에 IPC 정보가 포함되어 있다면 IPC 문자열을 이용해서 \
정보를 검색해 주세요"))

# prompt 의 messages를 출력합니다.
prompt.messages

In [ ]:
from langchain.agents import create_openai_functions_agent

# OpenAI 함수 기반 에이전트를 생성합니다.
# llm, tools, prompt를 인자로 사용합니다.
agent01 = create_openai_functions_agent(llm, tools01, prompt)

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.agents import AgentExecutor, create_react_agent

# session_id 를 저장할 딕셔너리 생성
store = {}

# session_id 를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    if session_ids not in store:  # session_id 가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환

agent_executor01 = AgentExecutor(
    agent=agent01,
    tools=tools01,
    verbose=False,
    handle_parsing_errors=True,
)

# 채팅 메시지 기록이 추가된 에이전트를 생성한다.
agent_with_chat_history01 = RunnableWithMessageHistory(
    agent_executor01,
    # 대화 session_id
    get_session_history,
    # 프롬프트의 질문이 입력되는 key: "input"
    input_messages_key="input",
    # 프롬프트의 메시지가 입력되는 key: "chat_history"
    history_messages_key="chat_history",
)

In [ ]:
!pip install -qU langchain-teddynote

In [ ]:
from langchain_teddynote.messages import AgentStreamParser
from langchain.agents.format_scratchpad.log import format_log_to_str

# 질의에 대한 답변을 스트리밍으로 출력 요청

intermediate_steps = []
response = agent_with_chat_history01.stream(
    { "input": "'지능정보 기술을 활용하여 사업장 내 유해인자 발생에 대한 정보를 제공하고, 서비스와 근로자의 질병위험도를 예측할 수 있는' 와 관련있는 특허를 알려주세요",
      "agent_scratchpad": format_log_to_str(intermediate_steps)
    },
    # session_id 설정
    config={"configurable": {"session_id": "abc123"}},
)

# 출력 확인
# 각 단계별 출력을 위한 파서 생성
agent_stream_parser = AgentStreamParser()
for step in response:
    agent_stream_parser.process_agent_steps(step)

In [ ]:
from langchain_teddynote.messages import AgentStreamParser
from langchain.agents.format_scratchpad.log import format_log_to_str

# 질의에 대한 답변을 스트리밍으로 출력 요청

intermediate_steps = []
response = agent_with_chat_history01.stream(
    { "input": "G16H-050/20,[A61B-005/00, A61B-005/021, A61B-005/024, A61B-005/11, A61B-005/145, G16H-010/20, G16H-010/60, G16H-020/00, G16H-040/20, G16H-080/00]` 인 정보를 알려줘",
      "agent_scratchpad": format_log_to_str(intermediate_steps)
    },
    # session_id 설정
    config={"configurable": {"session_id": "abc123"}},
)

# 출력 확인
# 각 단계별 출력을 위한 파서 생성
agent_stream_parser = AgentStreamParser()
for step in response:
    agent_stream_parser.process_agent_steps(step)

### 2. (방법2) 작은 토큰으로 나누어 2중으로 관리

####가. 구글드라이브 설정

In [ ]:
# 다운로드 데이터를 사용하기 위해 구글드라이브와 연결한다.
from google.colab import drive
drive.mount('/content/drive')

%cd /content/drive/MyDrive/Colab\ Notebooks/testdata/contest/
%ls

####나. 디스크에 저장된 벡터 DB가 존재할 경우 이부분은 SKIP 한다.

In [ ]:
# 토크나이저로 RecursiveCharacterTextSplitter() 를 사용한다.
# chunk 사이즈를 300 으로 해서 문서를 자른다.
from langchain.text_splitter import RecursiveCharacterTextSplitter

parent_text_splitter = RecursiveCharacterTextSplitter(chunk_size=600)
child_text_splitter = RecursiveCharacterTextSplitter(chunk_size=300)

In [ ]:
# - docs (1000개를 기준으로 자른 토큰)를 더 작은 단위(400개를 기준으로 함)로 쪼갠다.
# - uuid 를 서브청크와 메인청크 모두에 저장하여 서로 연동되도록 한다.
# - 여기서는 서브청구에 uuid 를 추가했다.
parent_docs = []
for i, doc in enumerate(docs):
    # 200크기로 자른다.
    _sub_docs = parent_text_splitter.split_documents([doc])
    for _doc in _sub_docs:    # 200 단위로 쪼겐 문단을 하나씩 불러온다.
        # metadata 에 새로운 항목 id_key를 생성하고 여기에 _id 값을 대입한다.
       _doc  = [Document(page_content=(_doc.page_content +  "; 메인IPC2 : " + 메인IPC2[i] +  "; 전체 IPC : " + 전체IPC[i] + "; 출원번호 : " + 출원번호[i] \
                                       + "; 등록번호 : " + 등록번호[i] + "; 출원인 : " + 출원인[i] + "; 발명자 : " + 발명자[i] + "; 법적상태 : " + 법적상태[i] \
                                       + "; " + id_key + " : " + doc_ids[i]), ids=doc_ids[i], \
                         metadata={id_key: doc_ids[i], "메인IPC2": 메인IPC2[i], "전체 IPC": 전체IPC[i], "출원번호": 출원번호[i]})]
       parent_docs.extend(_doc)

In [ ]:
# - docs (1000개를 기준으로 자른 토큰)를 더 작은 단위(400개를 기준으로 함)로 쪼갠다.
# - uuid 를 서브청크와 메인청크 모두에 저장하여 서로 연동되도록 한다.
# - 여기서는 서브청구에 uuid 를 추가했다.
child_docs = []
for i, doc in enumerate(docs):
    # 200크기로 자른다.
    _sub_docs = child_text_splitter.split_documents([doc])
    for _doc in _sub_docs:    # 200 단위로 쪼겐 문단을 하나씩 불러온다.
        # metadata 에 새로운 항목 id_key를 생성하고 여기에 _id 값을 대입한다.
       _doc  = [Document(page_content=(_doc.page_content + "; 메인IPC2 : " + 메인IPC2[i] +  "; 전체 IPC : " + 전체IPC[i] + "; 출원번호 : " + 출원번호[i] \
                                       + "; 등록번호 : " + 등록번호[i] + "; 출원인 : " + 출원인[i] + "; 발명자 : " + 발명자[i] + "; 법적상태 : " + 법적상태[i] \
                                       + "; " + id_key + " : " + doc_ids[i]), ids=doc_ids[i], \
                         metadata={id_key: doc_ids[i], "메인IPC2": 메인IPC2[i], "전체 IPC": 전체IPC[i], "출원번호": 출원번호[i]})]
       child_docs.extend(_doc)

In [ ]:
docs[:1]

In [ ]:
parent_docs[:2]

In [ ]:
child_docs[:2]

#### 다. VECTOR STORE 가지고 멀티 RETRIEVER 생성

In [ ]:
# vector db 를 메모리에 저장한다.
# Large Chunk 를 저장할 것이다.
# =====> 이부분 store 에 디렉토리를 지정하면 디스크에 저장됨

# 혹시 존재할 수 있는 기존 데이터를 모두 삭제한다.
#
# 만약 메모리에 vector DB를 사용하려면 InMemoryByteStore()을 사용하고
# 디스크를 사용하려면 LocalFileStore() 를 사용한다.
# store = InMemoryByteStore()
store = LocalFileStore("./store/smallizer")

id_key = "doc_id"

# [중요] 일반 Retriever 가 아니라 MultiVectorRetriever 를 선언하였다.
# - vectorstore 또는 docstore 에는 Large Chunk 를 저장하고
# - byte_store 에는 Small Chuhk 를 저장한다.
retriever02 = MultiVectorRetriever (
    vectorstore=vectorstore02,         # 자식 문서 즉 작은 Chunk가 저장된다.
    persist_directory="./store/smallizer",    # 만약 디스크에 영구 보관하고 싶다면 아래와 같이 지정한다.
    byte_store=store,                # 부모 문서 즉 큰 Chunk가 저장된다.
    id_key=id_key,                   # id_key 를 지정하여 상호 연결되도록 구성함
    k=10
)

# 만약 BYTE 단위가 아니라 DOC 단위로 저장하고 싶다면
# store = InMemoryStore()
# retriever = ParentDocumentRetriever(
#    vectorstore=vectorstore,
#    docstore=store,
#    child_splitter=child_splitter,
#    parent_splitter=parent_splitter
# )

# 일반적으로 사용했던
# vectorstore = Chroma.from_documents(documents=docs, embedding=embedding_model)
# retriever = vectorstore.as_retriever()
# 도 같이 기억해 둘것

####라. 벡터DB 에 데이터 저장 (만약 벡터DB 데이터가 존재할 경우 SKIP)

In [ ]:
# 작게 쪼갠 즉 400단위로 쪼갠 벡터를 vectstore 에 저장한다.
# 벡터 스토어는 아래와 같이 위에 정의되어 있다.
# vectorstore = Chroma(
#    collection_name="full_documents", embedding_function=OpenAIEmbeddings(openai_api_key=OPENAI_KEY)
# )

retriever02.vectorstore.add_documents(parent_docs)
retriever02.vectorstore.add_documents(child_docs)

# mset() 메서드를 통해 문서ID 와 문서 내용을 KEY-VALUE 쌍으로 문서 저장소에 저장한다.
# docstore 로 저장하게 되면 byte_store 로 지정된 영역에 저장된다.
retriever02.docstore.mset(list(zip(doc_ids, docs)))

In [ ]:
retriever02.invoke("전체IPC 가 G16H-010/60,\[G06Q-010/10, G16H-010/20, G16H-080/00\]' 인 특허를 알려주세요")

In [ ]:
retriever02.invoke("'지능정보 기술을 활용하여 사업장 내 유해인자 발생에 대한 정보를 제공하고, 서비스와 근로자의 질병위험도를 예측할 수 있는' 와 관련 있는 특허를 알려주세요")

####마. Agent 생성

In [ ]:
from langchain_openai import ChatOpenAI

# ChatOpenAI 클래스를 langchain_openai 모듈에서 가져옵니다.
llm = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0, openai_api_key=OPENAI_KEY)

In [ ]:
# 사용자 정의가 아닌 외부 도구에 의한 TOOL 은 아래와 같이 정의한다.

from langchain.tools.retriever import create_retriever_tool

retriever_tool02 = create_retriever_tool(
    retriever02,
    # 함수이름은 반드시 영문자로 해야 한다. 한글로 하면 에러가 발생한다.
    name="Patent_inquiry02",
    description="사용자의 특허 관련 질문을 검색합니다. 만약 여기에 존재하지 않는 특허와 관련한 질문이라면 LLM을 통해 \
검색합니다. 만약 사용자의 질문에 IPC가 포함되어 있다면 반드시 여기에서 얻은 답변에도 동일한 IPC가 포함되어야 합니다."
)

In [ ]:
# tools 리스트에 search와 retriever_tool을 추가합니다.
tools02 = [retriever_tool02]

In [ ]:
from langchain import hub
from langchain.prompts.chat import SystemMessagePromptTemplate

# hub에서 prompt를 가져옵니다 - 이 부분을 수정할 수 있습니다!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.append(SystemMessagePromptTemplate.from_template("만약 사용자의 질문에 IPC 정보가 포함되어 있다면 IPC 문자열을 이용해서 \
정보를 검색해 주세요"))

# prompt 의 messages를 출력합니다.
prompt.messages

In [ ]:
from langchain.agents import create_openai_functions_agent

# OpenAI 함수 기반 에이전트를 생성합니다.
# llm, tools, prompt를 인자로 사용합니다.
agent = create_openai_functions_agent(llm, tools02, prompt)

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.agents import AgentExecutor, create_react_agent

# session_id 를 저장할 딕셔너리 생성
store = {}

# session_id 를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    if session_ids not in store:  # session_id 가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환

agent_executor02 = AgentExecutor(
    agent=agent,
    tools=tools02,
    verbose=False,
    handle_parsing_errors=True,
)

# 채팅 메시지 기록이 추가된 에이전트를 생성한다.
agent_with_chat_history02 = RunnableWithMessageHistory(
    agent_executor02,
    # 대화 session_id
    get_session_history,
    # 프롬프트의 질문이 입력되는 key: "input"
    input_messages_key="input",
    # 프롬프트의 메시지가 입력되는 key: "chat_history"
    history_messages_key="chat_history",
)

In [ ]:
!pip install -qU langchain-teddynote

In [ ]:
from langchain_teddynote.messages import AgentStreamParser
from langchain.agents.format_scratchpad.log import format_log_to_str

# 질의에 대한 답변을 스트리밍으로 출력 요청

intermediate_steps = []
response = agent_with_chat_history02.stream(
    { "input": "`G06Q-050/18,[G06F-017/18, G06N-003/00]` 인 특허를 알려줘",
      "agent_scratchpad": format_log_to_str(intermediate_steps)
    },
    # session_id 설정
    config={"configurable": {"session_id": "abc123"}},
)

# 출력 확인
# 각 단계별 출력을 위한 파서 생성
agent_stream_parser = AgentStreamParser()
for step in response:
    agent_stream_parser.process_agent_steps(step)

In [ ]:
from langchain_teddynote.messages import AgentStreamParser
from langchain.agents.format_scratchpad.log import format_log_to_str

# 질의에 대한 답변을 스트리밍으로 출력 요청

intermediate_steps = []
response = agent_with_chat_history02.stream(
    { "input": "G16H-050/20,[A61B-005/00, A61B-005/021, A61B-005/024, A61B-005/11, A61B-005/145, G16H-010/20, G16H-010/60, G16H-020/00, G16H-040/20, G16H-080/00]` 인 정보를 알려줘",
      "agent_scratchpad": format_log_to_str(intermediate_steps)
    },
    # session_id 설정
    config={"configurable": {"session_id": "abc123"}},
)

# 출력 확인
# 각 단계별 출력을 위한 파서 생성
agent_stream_parser = AgentStreamParser()
for step in response:
    agent_stream_parser.process_agent_steps(step)

###3. 메타정보를 이용하여 Retriever 생성

#### 가. 벡터 DB생성 (만약 저장된 DB가 있다면 이 작업은 SKIP)

In [ ]:
vectorstore03 = Chroma.from_documents(documents=docs, embedding=model_huggingface, persist_directory="./store/meta", collection_name="meta")

####나. 메타기반 retriever 생성

In [ ]:
from langchain_openai import ChatOpenAI

# ChatOpenAI 클래스를 langchain_openai 모듈에서 가져옵니다.
llm = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0, openai_api_key=OPENAI_KEY)

In [ ]:
from langchain.retrievers import SelfQueryRetriever
from langchain.retrievers import EnsembleRetriever
from langchain.retrievers.document_compressors import LLMChainFilter
from langchain.retrievers.document_compressors import LLMChainExtractor
from langchain.chains.query_constructor.base import AttributeInfo
import lark

# 메타데이터 필드 정보 생성
metadata_field_info = [
    AttributeInfo(
        name="doc_id",
        description="doc_id 는 이 문서를 다른 문서와 연결해 주는 key 값이다",
        type="string",
    ),
    AttributeInfo(
        name="메인IPC2",
        description="특허에서 주요 주제에 해당하는 IPC 코드이다.",
        type="string",
    ),
    AttributeInfo(
        name="전체 IPC",
        description="특허의 요약내용을 토대로 이 특허에 해당하는 IPC 전체 값을 가지고 있다.\
        사용자 요청이 IPC를 포함하고 있을 때는 반드시 '전체 IPC'를 이용하여 정확히 일치하는 데이터를 먼저 검색하고 \
        그 다음에 메인IPC2를 검색한다.",
        type="string",
    ),
    AttributeInfo(
        name="출원번호",
        description="특허의 출원번호를 저장하고 있다.",
        type="string",
    ),
     AttributeInfo(
        name="등록번호",
        description="특허의 등록번호를 저장하고 있다.",
        type="string",
    ),
    AttributeInfo(
        name="출원인",
        description="특허를 출원한 회사 또는 개인 정보를 저장하고 있다.",
        type="string",
    ),
    AttributeInfo(
        name="발명자",
        description="특허를 발명한 자의 정보를 보관하고 있다.",
        type="string",
    ),
    AttributeInfo(
        name="법적상태",
        description="특허를 현재 상태 정보를 보관하고 있다.",
        type="string",
    ),
]

from langchain.retrievers.multi_query import MultiQueryRetriever

# LLM 을 사용하여 쿼리를 변환하고 메나데이터 필터링을 수행한다.
retriever03 = SelfQueryRetriever.from_llm (
    llm=llm,
    vectorstore=vectorstore03,
    metadata_field_info=metadata_field_info,
    document_contents="특허 정보를 가지고 있다",
    verbose=True
)

####다. Retriever 의 정확성 검증

In [ ]:
retriever03.invoke("전체 IPC 가 G16H-010/60,\[G06Q-010/10, G16H-010/20, G16H-080/00\] 와 정확히 일치하는 특허를 알려주세요")

In [ ]:
retriever03.invoke('''"기능성위장관질환의 증상 조절을 위한 자가완성형 음식 조절 서비스 제공 시스템 및
그의 음식 조절 서비스 제공 방법이 제공된다. 사용자 단말기는 사용자의 증상에 따라 음식 조절 가이드라인을 제공하는 음식 조절 어플리케이션이
사용자 별 상세 서비스 제공 모드로 동작하면, 음식 조절 어플리케이션을 통해 사용자의 기능성 위장관 증상 및 기능성 위장관 증상이 발생하기 전에 섭취한 음식 정보
(이하, '사용자별 증상 유발 음식 정보'라 한다)를 사용자로부터 입력받고, 질환 진단 및 음식 조절 서버는 사용자 단말기로부터 상기 기능성 위장관 증상 및 사용자별 증상 유발 음식
정보가 수신되면, 수신된 기능성 위장관 증상을 분석하여 사용자의 기능성 위장관 질환을 진단하고, 수신된 사용자별 증상 유발 음식 정보를 분석하여 진단된 기능성 위장관 질환을 예방하기
위한 음식 조절 가이드라인을 작성하여 사용자 단말기에게 제공한다." 와 관련한 특허 정보를 알려주세요''')

In [ ]:
# 요약 데이터로 부터 질문에 대한 답을 가져온다.
retriever03.invoke("인공지능과 관련한 특허 3개만 50자 이내로 요약해줘?")

In [ ]:
retriever03.invoke("메인IPC2 G06Q 인 특허 1개만 알려줘 ?")

In [ ]:
retriever03.invoke("전체 IPC 가 G16H-010/60,\[G06Q-010/10, G16H-010/20, G16H-080/00\]' 와 정확히 일치하는 특허를 알려주세요")

In [ ]:
retriever03.invoke("전체 IPC 가 G16H-010/60,\[G06Q-010/10, G16H-010/20, G16H-080/00\]' 인 특허를 알려주세요")

In [ ]:
retriever03.invoke("전체 IPC 가 `G16H-020/30,[A63B-024/00, G16H-010/60, G16H-040/20]` 정확히 일치하는 특허를 알려줘")

####[참고] 참고 코드임 (search tool 을 생성함)

In [ ]:
# Tavily API 키도 Colab '보안 비밀' 또는 환경변수(TAVILY_API_KEY)에서 읽어온다.
import os

try:
    from google.colab import userdata
    os.environ["TAVILY_API_KEY"] = userdata.get("TAVILY_API_KEY")
except Exception:
    assert "TAVILY_API_KEY" in os.environ, "TAVILY_API_KEY 환경변수를 설정하세요"

In [ ]:
!pip install python-dotenv

In [ ]:
# TavilySearchResults 클래스를 langchain_community.tools.tavily_search 모듈에서 가져온다.
from langchain_community.tools.tavily_search import TavilySearchResults

# TavilySearchResults 클래스의 인스턴스를 생성한다
# k=5은 검색 결과를 5개까지 가져오겠다는 의미다
search = TavilySearchResults(k=5)     # k 또는 max_result 사용

###[참고] Retriever Agent 생성 및 실행

In [ ]:
# 사용자 정의가 아닌 외부 도구에 의한 TOOL 은 아래와 같이 정의한다.

from langchain.tools.retriever import create_retriever_tool

retriever_tool = create_retriever_tool(
    retriever01,
    # 함수이름은 반드시 영문자로 해야 한다. 한글로 하면 에러가 발생한다.
    name="Patent_inquiry",
    description=" 사용자가 입력한 특허 관련 문장을 실행합니다. \
    1. 만약 사용자의 질문에 IPC 정보가 포함되어 있다면 반드시 TOOL 에게 IPC 정보가 정확하게 일치하는 정보를 찾아달라고 요청한다. \
       추출된 정보가 없을 경우에는 IPC를 기준으로 정보를 추출합니다. \
    2. 만약 사용자의 질문에 IPC 정보가 포함되어 있지 않을 경우 사용자의 질문에 따라 동작합니다. "
)

In [ ]:
# tools 리스트에 search와 retriever_tool을 추가합니다.
tools = [retriever_tool]

In [ ]:
from langchain import hub
from langchain.prompts.chat import SystemMessagePromptTemplate

# hub에서 prompt를 가져옵니다 - 이 부분을 수정할 수 있습니다!
prompt = hub.pull("hwchase17/openai-functions-agent")
prompt.append(SystemMessagePromptTemplate.from_template(
    " 당신은 특허 전문가 입니다. \
     1. 사용자의 질문에 IPC 정보가 포함되어 있다면 '전체 IPC' 가 정확히 일치하는 정보를 tools 에서 검색합니다. \
        아울러 출력에는 전체 IPC 가 포함되어야 합니다. \
     2. 사용자의 질문에 IPC 정보가 포함되지 않았다면 유사한 특허정보를 tools 에서 검색합니다.   \
     3. 만약 해당 정보가 존재하지 않는다면 당신이 직접 해당 정보를 생성하여 제공합니다."))

# prompt 의 messages를 출력합니다.
prompt.messages

In [ ]:
from langchain.agents import AgentExecutor, create_react_agent, create_openai_tools_agent

# OpenAI 함수 기반 에이전트를 생성합니다.
# llm, tools, prompt를 인자로 사용합니다.
agent = create_openai_tools_agent(llm, tools, prompt)

In [ ]:
from langchain_community.chat_message_histories import ChatMessageHistory
from langchain_core.runnables.history import RunnableWithMessageHistory
from langchain.agents import AgentExecutor, create_react_agent

# session_id 를 저장할 딕셔너리 생성
store = {}

# session_id 를 기반으로 세션 기록을 가져오는 함수
def get_session_history(session_ids):
    if session_ids not in store:  # session_id 가 store에 없는 경우
        # 새로운 ChatMessageHistory 객체를 생성하여 store에 저장
        store[session_ids] = ChatMessageHistory()
    return store[session_ids]  # 해당 세션 ID에 대한 세션 기록 반환

agent_executor = AgentExecutor(
    agent=agent,
    tools=tools,
    verbose=False,
    handle_parsing_errors=True,
)

# 채팅 메시지 기록이 추가된 에이전트를 생성한다.
agent_with_chat_history = RunnableWithMessageHistory(
    agent_executor,
    # 대화 session_id
    get_session_history,
    # 프롬프트의 질문이 입력되는 key: "input"
    input_messages_key="input",
    # 프롬프트의 메시지가 입력되는 key: "chat_history"
    history_messages_key="chat_history",
)

In [ ]:
!pip install -qU langchain-teddynote

In [ ]:
from langchain_teddynote.messages import AgentStreamParser
from langchain.agents.format_scratchpad.log import format_log_to_str

# 질의에 대한 답변을 스트리밍으로 출력 요청
intermediate_steps = []

def query(input):
   response = agent_with_chat_history.stream (
              { "input": input,
                "agent_scratchpad": format_log_to_str(intermediate_steps),
              },
              # session_id 설정
              config={"configurable": {"session_id": "abc123"}},
   )

   # 출력 확인
   # 각 단계별 출력을 위한 파서 생성
   agent_stream_parser = AgentStreamParser()
   for step in response:
        agent_stream_parser.process_agent_steps(step)

####라. Chain 생성 및 실행

In [ ]:
llm = ChatOpenAI(model="gpt-4-turbo-preview", temperature=0, openai_api_key=OPENAI_KEY)

In [ ]:
from langchain.memory  import ConversationBufferMemory
from langchain.prompts import ChatPromptTemplate,  MessagesPlaceholder, SystemMessagePromptTemplate, HumanMessagePromptTemplate
from langchain import PromptTemplate
from langchain_core.runnables import RunnableBranch, RunnablePassthrough
from langchain.chains import LLMChain
from langchain_core.runnables import chain
from langchain.schema import StrOutputParser
from operator  import itemgetter

@chain
def format_docs(docs):
    # string = "\n".join([doc.metadata for doc in docs])
    return("\n".join([doc.page_content for doc in docs]))

@chain
def query(_dict):
  _dict["input"] = _dict["input"] + "단, 가장 일치하는 정보를 검색해 주세요"
  return _dict["input"]

def question(input):
    사용자질문 = {"input" : input}
    global result
    result = final_chain.invoke(사용자질문)
    # default 체인(LLMChain)은 dict 를 반환하므로 text 만 꺼낸다.
    if isinstance(result, dict):
        result = result.get("text", result)

search_template = """ 당신은 특허 전문가 입니다. \
             당신은 아래에 '제공되는 정보'를 기준으로 정보를 제공합니다. \
             요청과 일치하는 정보를 제공해 주세요 \
             아울러 아래의 '제공되는 정보' 에서 '출원번호' '메인IPC' \
             '전체IPC', '요약정보', '법적상태', '출원인', '등록번호' 정보를 추출합니다. \
             만약 정보를 '제공되는 정보'에서 추출하지 못할 경우 '검색불가' 로 기재합니다. \
             아울러 '제공되는 정보' 가 없을 경우에는 당신이 생성합니다. \
             만약 추출한 '요약정보'가 100자 이상일 경우 이를 50자 이내로 요약합니다. \
            '제공되는 정보' : {context} \
             출력시에 아래의 규칙에 따라 출력해 줍니다. \
            '출원번호' :    \
            '메인IPC'  :    \
            '전체IPC'  :    \
            '요약정보' :    \
            '법적상태' :    \
            '등록번호' :    \
            '출원인'   :       """

generator_template = """ 당신은 특허코드인 IPC 를 생성해 주는 전문가 입니다.
    당신은 사용자가 입력한 {input} 을 가지고 국제 기준에 맞는 특허 IPC 코드를 생성합니다.
    아래와 같은 작업을 단계별로 수행합니다.
    1. 가장 중요한 내용 순으로 정보를 추출합니다.
    2. 각 추출된 내용을 가지고 IPC 코드, 한글코드, 사유를 생성합니다. 최대 10개까지 생성합니다.
    3. 아래의 출력 형식을 사용하여 출력합니다.
    [출력형식]
    1. 만약 문장이 "다." 로 끝나면 한줄 띄어쓰기를 합니다.
    2. 추출된 정보를 중요도 순서대로 기재합니다.
    3. 요약된 내용은 "요약내용 : " 형태로 출력합니다.
    4. IPC :   한글코드 :    =====> 코드 부여 이유 \n 형태로 출력합니다.
    """

prompt_infos = [
    {
        "name": "search",
        "description": "기존 정보를 조회합니다",
        "prompt_template": search_template,
    },
    {
        "name": "generator",
        "description": "새로운 정보를 생성합니다.",
        "prompt_template": generator_template,
    },
]

destination_chains = {}

# 정의된 프롬프트들로 부터 하나의 프롬프트씩 꺼낸다.
for prompt_info in prompt_infos:

    # 꺼낸 프롬프트로 부터 이름을 추출하여 이름 변수에 대입한다.
    name = prompt_info["name"]

    # 꺼낸 프롬프트로 부터 prompt_template 를 추출하여 prompt 변수에 대입한다.
    prompt = PromptTemplate.from_template(prompt_info["prompt_template"])

    if "search" in name:
        chain = {"context" : query | retriever03 | format_docs, "question" : RunnablePassthrough()} | prompt | llm | StrOutputParser()
    else:
        chain = prompt | llm |  StrOutputParser()

    destination_chains[name] = chain

# 체인의 이름과 description 으로 구성된 리스트변수를 생성한다.
destinations = [f"{p['name']}: {p['description']}" for p in prompt_infos]

# 세개의 체인을 "\n"으로 구분하여 연결한다.
destinations_str = "\n".join(destinations)
print("연결된 destinations_str 은 ", destinations_str)

# "{input}" 하나만 있는 프롬프트를 생성한다.
default_prompt = PromptTemplate.from_template("{input}")

# 바로 위에서 생성한 프롬프트를 가지고 default_chain 을 생성한다.
default_chain = LLMChain(llm=llm, prompt=default_prompt)

router_prompt = PromptTemplate(
    template="""
For the given question, refer to the explanations for the following categories below and select the appropriate category.
If there is no suitable category, return the string "default".
The output should be single word

Categories are composed of strings in the format "Category: Description", and the categories are as follows:
{destinations_str}

Question:
{input}
""",
    input_variables=["input"],
    partial_variables={"destinations_str": destinations_str},
)

router_chain = {"input": itemgetter("input")} | router_prompt | llm | StrOutputParser()

branch = RunnableBranch(
    (lambda x: "search" in x['destination'].lower(), destination_chains['search']),
    (lambda x: "generator" in x['destination'].lower(), destination_chains['generator']),
    default_chain,
 )

final_chain = {"input": itemgetter("input"), "destination": router_chain} | branch

In [ ]:
from langchain_core.runnables import RunnableLambda

router01_chain = {"input" : itemgetter("input"), "destination": router_chain} | RunnablePassthrough()

In [ ]:
router01_chain.invoke({"input" : "정보를 찾아주세요"})

In [ ]:
question("""기능성위장관질환의 증상 조절을 위한 자가완성형 음식 조절 서비스 제공 시스템 및 \
그의 음식 조절 서비스 제공 방법이 제공된다. 사용자 단말기는 사용자의 증상에 따라 음식 조절 가이드라인을 제공하는 음식 조절 어플리케이션이 \
사용자 별 상세 서비스 제공 모드로 동작하면, 음식 조절 어플리케이션을 통해 사용자의 기능성 위장관 증상 및 기능성 위장관 증상이 발생하기 전에 섭취한 음식 정보 \
(이하, '사용자별 증상 유발 음식 정보'라 한다)를 사용자로부터 입력받고, 질환 진단 및 음식 조절 서버는 사용자 단말기로부터 상기 기능성 위장관 증상 및 사용자별 증상 유발 음식 \
정보가 수신되면, 수신된 기능성 위장관 증상을 분석하여 사용자의 기능성 위장관 질환을 진단하고, 수신된 사용자별 증상 유발 음식 정보를 분석하여 진단된 기능성 위장관 질환을 예방하기\
위한 음식 조절 가이드라인을 작성하여 사용자 단말기에게 제공한다." 와 관련한 특허 정보를 알려주세요""")

In [ ]:
print(result)

In [ ]:
question("전체 IPC 가 'G06Q-050/18,[G06F-017/18, G06N-003/00]' 인 정보를 알려줘")

In [ ]:
print(result)

In [ ]:
question("출원인이 '티쓰리큐 주식회사' 인 특허를 3개 알려줘")

In [ ]:
print(result)

In [ ]:
question('''"기능성위장관질환의 증상 조절을 위한 자가완성형 음식 조절 서비스 제공 시스템 및 \
그의 음식 조절 서비스 제공 방법이 제공된다. 사용자 단말기는 사용자의 증상에 따라 음식 조절 가이드라인을 제공하는 음식 조절 어플리케이션이 \
사용자 별 상세 서비스 제공 모드로 동작하면, 음식 조절 어플리케이션을 통해 사용자의 기능성 위장관 증상 및 기능성 위장관 증상이 발생하기 전에 섭취한 음식 정보 \
(이하, '사용자별 증상 유발 음식 정보'라 한다)를 사용자로부터 입력받고, 질환 진단 및 음식 조절 서버는 사용자 단말기로부터 상기 기능성 위장관 증상 및 사용자별 증상 유발 음식 \
정보가 수신되면, 수신된 기능성 위장관 증상을 분석하여 사용자의 기능성 위장관 질환을 진단하고, 수신된 사용자별 증상 유발 음식 정보를 분석하여 진단된 기능성 위장관 질환을 예방하기 \
위한 음식 조절 가이드라인을 작성하여 사용자 단말기에게 제공한다." 와 관련한 IPC를 생성해주세요''')

In [ ]:
print(result)